# Claim Documents: `ai_parse_document` -> `ai_query`

SQL-native PDF extraction chain. The parser emits layout-aware text (with OCR
for image-only pages), and `ai_query` emits typed JSON against a shared schema.

**Why this path.** `ai_query` does not accept PDF bytes today — Anthropic/Databricks
confirmed the gap in April 2026 (Alice Li: *"ai_query isn't meant to be for parsing
PDF files ... we ask customers to use ai_parse_document for the pdf use cases"*;
Andrew Shieh, 2026-04-07: *"we don't currently have support for PDF input types with
ai_query for any model ... options under discussion"*). So on Databricks today the two
supported PDF paths are:

1. `ai_parse_document` → `ai_query` (**this notebook**) — SQL-native, governable,
   reuses the parser's layout/OCR output.
2. FMAPI DocumentContent via REST/SDK with Ray parallelism — see
   [`claim_doc_ray_claude.ipynb`](claim_doc_ray_claude.ipynb).

Both are scored against the same hand-labeled golden set of 10 public insurance docs.

## 1. Setup

In [ ]:
%pip install --quiet mlflow requests
%restart_python

In [ ]:
import json, requests
import pandas as pd
from time import perf_counter
import mlflow

CATALOG, SCHEMA, VOLUME = "shm", "genai", "raw_pdfs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
MODEL = "databricks-claude-sonnet-4"

user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/document_intelligence_eval")

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

sample_docs = {
    "acord_certificate.pdf":       "https://azdot.gov/sites/default/files/2019/06/sample-insurance-certificate-accord-form.pdf",
    "acord_nyc_dcla.pdf":          "https://www.nyc.gov/assets/dcla/downloads/pdf/cdf_sample_cgl.pdf",
    "acord_nyc_mome.pdf":          "https://www.nyc.gov/assets/mome/pdf/Sample-MOME-ACORD_25_2016-03.pdf",
    "nyc_workerscomp_sample.pdf":  "https://www.nyc.gov/html/dcla/downloads/pdf/cdf_sample_workerscomp_c_105_2.pdf",
    "acord_nyc_dycd.pdf":          "https://www.nyc.gov/assets/dycd/downloads/pdf/Insurance-Sample-Package25.pdf",
    "acord_moval.pdf":             "https://moval.gov/departments/media/pdf/SampleInsuranceCertsEndorsements.pdf",
    "acord_ny_ogs.pdf":            "https://ogs.ny.gov/system/files/documents/2018/10/acord25version918dcsamplenopollution.pdf",
    "nyc_acord_sample.pdf":        "https://www.nyc.gov/assets/buildings/pdf/acord_cert_of_ins_sample.pdf",
    "cms1500_claim_form.pdf":      "https://www.cms.gov/medicare/cms-forms/cms-forms/downloads/cms1500.pdf",
    "acord_tx.pdf":                "https://www.rrc.texas.gov/media/kh2p1ugz/acord25_-certificate-of-insurance.pdf",
}
for filename, url in sample_docs.items():
    try:
        resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        with open(f"{VOLUME_PATH}/{filename}", "wb") as f:
            f.write(resp.content)
    except Exception:
        pass  # acord_tx.pdf SSL fails — assume pre-loaded

display(spark.sql(f"LIST '{VOLUME_PATH}'"))

### Shared prompt + schema

Two non-obvious rules, learned empirically: **placeholders are not values** (forms
contain template text like "Insert Insurer name"), and **primary certificate only**
(multi-page packages mix real certs with instructions).

In [ ]:
EXTRACTION_PROMPT = """Extract structured fields from this insurance document. Return null for missing fields.

RULES:
1. Placeholders are not values. Template text like "Insert Insurer name", "UNDERWRITER NAME REQUIRED", "Carrier A", "Grantee Organization", "CBO NAME" must return null — not the literal placeholder.
2. Primary certificate only. If the document contains multiple certificates, endorsements, or instruction pages, use the first/primary certificate.
3. Dates in MM/DD/YYYY when possible.

Fields: document_type, insurer_name, insured_name, policy_number, effective_date, expiration_date, coverage_types (list), total_amount, key_tables.
"""

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {"name": "claim_extraction", "schema": {
        "type": "object",
        "properties": {
            "document_type":   {"type": ["string", "null"]},
            "insurer_name":    {"type": ["string", "null"]},
            "insured_name":    {"type": ["string", "null"]},
            "policy_number":   {"type": ["string", "null"]},
            "effective_date":  {"type": ["string", "null"]},
            "expiration_date": {"type": ["string", "null"]},
            "coverage_types":  {"type": "array", "items": {"type": "string"}},
            "total_amount":    {"type": ["string", "null"]},
            "key_tables":      {"type": ["string", "null"]},
        },
        "required": ["document_type", "insurer_name", "insured_name", "policy_number",
                     "effective_date", "expiration_date", "coverage_types", "total_amount", "key_tables"],
    }},
}
RESPONSE_SCHEMA_STR = json.dumps(RESPONSE_SCHEMA)

## 2. `ai_parse_document` -> `ai_query`

`ai_parse_document` emits a VARIANT with layout-aware elements (incl. OCR for
image-only pages); flatten `elements[].content` to text and feed it to `ai_query`
with the shared prompt + `responseFormat`.

In [ ]:
for tbl in ["parsed_docs", "aiquery_extractions", "benchmark_results"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{tbl}")

t0 = perf_counter()
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SCHEMA}.parsed_docs AS
    SELECT regexp_extract(path, '([^/]+)$', 1) AS filename,
           ai_parse_document(content) AS parsed
    FROM read_files('{VOLUME_PATH}')
""")
parse_wall_s = perf_counter() - t0

prompt_sql = EXTRACTION_PROMPT.replace("'", "''")
schema_sql = RESPONSE_SCHEMA_STR.replace("'", "''")

t0 = perf_counter()
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SCHEMA}.aiquery_extractions AS
    WITH parsed_text AS (
        SELECT filename,
               CONCAT_WS('\\n',
                   TRANSFORM(
                       from_json(to_json(parsed:document:elements), 'ARRAY<STRUCT<content:STRING,type:STRING>>'),
                       x -> x.content
                   )
               ) AS full_text
        FROM {CATALOG}.{SCHEMA}.parsed_docs
    )
    SELECT filename,
           ai_query('{MODEL}',
                    CONCAT('{prompt_sql}', '\\n\\nDOCUMENT TEXT:\\n', full_text),
                    responseFormat => '{schema_sql}') AS extracted
    FROM parsed_text
""")
chain_wall_s = perf_counter() - t0

print(f"parse wall: {parse_wall_s:.1f}s   chain wall: {chain_wall_s:.1f}s")
display(spark.table(f"{CATALOG}.{SCHEMA}.aiquery_extractions"))

## 3. Golden set + tolerant scorer

Substring-tolerant list match (`General Liability` ~ `Commercial General Liability`)
and YY↔YYYY date normalization. Strict exact-match hides real accuracy under
formatting noise.

In [ ]:
import re

GENERIC_ACORD_COVERAGES = ["General Liability", "Automobile Liability", "Umbrella Liability", "Workers Compensation"]
CMS1500_PAYORS = ["Medicare", "Medicaid", "TRICARE", "CHAMPVA"]

golden_set = [
    {"filename": "acord_certificate.pdf",      "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": "Sample company", "insured_name": "Sample Insured Inc", "policy_number": "Sample no",
     "effective_date": "01/01/2010", "expiration_date": "01/01/2011", "coverage_types": GENERIC_ACORD_COVERAGES},
    {"filename": "acord_nyc_dcla.pdf",         "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": "US Underwriters Insurance Co", "insured_name": None, "policy_number": "ABCD1234567",
     "effective_date": "07/01/2025", "expiration_date": "06/30/2026",
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "acord_nyc_mome.pdf",         "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": "123456789",
     "effective_date": None, "expiration_date": None,
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "nyc_workerscomp_sample.pdf", "document_type": "NYS Workers Compensation Certificate",
     "insurer_name": "ABC Insurance Company", "insured_name": None, "policy_number": "E 1234567890",
     "effective_date": "07/01/2016", "expiration_date": "06/30/2017",
     "coverage_types": ["Workers Compensation"]},
    {"filename": "acord_nyc_dycd.pdf",         "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None,
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "acord_moval.pdf",            "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": "XX0111222333",
     "effective_date": None, "expiration_date": None,
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Workers Compensation"]},
    {"filename": "acord_ny_ogs.pdf",           "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": "04/01/2011", "expiration_date": "04/01/2012",
     "coverage_types": ["Commercial General Liability", "Automobile Liability", "Umbrella Liability", "Workers Compensation"]},
    {"filename": "nyc_acord_sample.pdf",       "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None, "coverage_types": GENERIC_ACORD_COVERAGES},
    {"filename": "cms1500_claim_form.pdf",     "document_type": "CMS-1500 Health Insurance Claim Form",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None, "coverage_types": CMS1500_PAYORS},
    {"filename": "acord_tx.pdf",               "document_type": "ACORD Certificate of Liability Insurance",
     "insurer_name": None, "insured_name": None, "policy_number": None,
     "effective_date": None, "expiration_date": None, "coverage_types": GENERIC_ACORD_COVERAGES},
]

EVAL_FIELDS = ["document_type", "insurer_name", "insured_name", "policy_number",
               "effective_date", "expiration_date", "coverage_types"]

def _norm(s):
    return str(s).lower().strip().replace("commercial ", "").replace("'", "").replace("  ", " ")

def _list_match(exp, ext):
    if not exp and not ext: return 1.0
    if not exp or not ext:  return 0.0
    e, x = [_norm(v) for v in exp], [_norm(v) for v in ext]
    matches = sum(1 for v in e if any(v in w or w in v for w in x))
    return max(0.0, matches / len(e) - min(0.2, 0.05 * max(0, len(x) - matches)))

def _parse_date(s):
    m = re.match(r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2,4})", str(s).strip())
    if not m: return None
    mo, d, y = int(m[1]), int(m[2]), int(m[3])
    y += 2000 if y < 50 else 1900 if y < 100 else 0
    return (mo, d, y)

def _date_match(exp, ext):
    e, x = _parse_date(exp), _parse_date(ext)
    return 1.0 if e and x and e == x else 0.0

def score_field(ext_v, exp_v, field=None):
    if exp_v is None and ext_v in (None, "", "<UNKNOWN>", "null"): return 1.0
    if exp_v is None or ext_v is None: return 0.0
    if isinstance(exp_v, list): return _list_match(exp_v, ext_v if isinstance(ext_v, list) else [])
    if field in ("effective_date", "expiration_date"): return _date_match(exp_v, ext_v)
    e, x = _norm(exp_v), _norm(ext_v)
    return 1.0 if e == x else 0.8 if (e in x or x in e) else 0.0

def score_extraction(extracted, expected):
    s = {f: score_field(extracted.get(f), expected.get(f), f) for f in EVAL_FIELDS}
    s["accuracy"] = sum(s.values()) / len(EVAL_FIELDS)
    return s

### Benchmark summary

In [ ]:
golden = {g["filename"]: g for g in golden_set}
rows = []
for r in spark.sql(f"SELECT filename, extracted FROM {CATALOG}.{SCHEMA}.aiquery_extractions").collect():
    if not r["extracted"] or r["filename"] not in golden:
        continue
    try:
        sc = score_extraction(json.loads(r["extracted"]), golden[r["filename"]])
    except json.JSONDecodeError:
        continue
    rows.append({"filename": r["filename"], **sc})

bench_df = pd.DataFrame(rows)
spark.createDataFrame(bench_df).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.benchmark_results")

wall_s = parse_wall_s + chain_wall_s
summary = bench_df[["accuracy"] + EVAL_FIELDS].mean().round(3).rename("mean").to_frame().T
print(f"wall: {wall_s:.1f}s   docs/s: {len(golden_set) / wall_s:.2f}")

with mlflow.start_run(run_name="aiquery_chain"):
    mlflow.log_param("model", MODEL)
    mlflow.log_param("approach", "ai_parse_document->ai_query")
    mlflow.log_metric("mean_accuracy", float(bench_df["accuracy"].mean()))
    mlflow.log_metric("parse_wall_s", float(parse_wall_s))
    mlflow.log_metric("chain_wall_s", float(chain_wall_s))
    mlflow.log_metric("total_wall_s", float(wall_s))
    mlflow.log_metric("docs_per_s", float(len(golden_set) / wall_s))
    mlflow.log_table(bench_df, "benchmark_detail.json")

display(summary)

## 4. Prompt registry

Delta-backed versioning so `ai_query` can pull the active prompt by name at query
time — full audit trail, rollback via `is_active` flag.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.prompt_registry")
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SCHEMA}.prompt_registry (
        prompt_name STRING, version INT, prompt_text STRING,
        response_schema STRING, model STRING,
        created_by STRING, created_at TIMESTAMP, is_active BOOLEAN
    ) USING DELTA
""")

prompt_sql = EXTRACTION_PROMPT.replace("'", "''")
schema_sql = RESPONSE_SCHEMA_STR.replace("'", "''")
spark.sql(f"""
    INSERT INTO {CATALOG}.{SCHEMA}.prompt_registry VALUES
        ('claim_extraction', 1, '{prompt_sql}', '{schema_sql}', '{MODEL}',
         current_user(), current_timestamp(), true)
""")

display(spark.sql(f"""
    SELECT prompt_name, version, model, is_active, created_by, created_at
    FROM {CATALOG}.{SCHEMA}.prompt_registry
"""))